# 07 · torch.compile and fused attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/07_compile_and_sdpa.ipynb) &nbsp; 📖 Concept deck: [Performance: compile, SDPA, memory](https://nilesh-salpe.github.io/visual-ai/pytorch/07-performance/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** compiling a function with `torch.compile`, checking it matches eager mode, finding graph breaks with `torch._dynamo.explain`, and measuring how attention memory grows with sequence length.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

import time
def gelu_mlp(x, w1, w2):
    return F.gelu(x @ w1) @ w2

## Task 1: Compile and compare with eager

Compile `gelu_mlp` with `torch.compile` and return the maximum absolute difference between compiled and eager outputs on random inputs (x: 64×128, w1: 128×512, w2: 512×128). The first call compiles (slow); later calls reuse the compiled code.

In [ ]:
def compiled_matches():
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_compiled_matches(f):
    d = f()
    assert d < 1e-4, f"compiled and eager outputs should match closely, max diff {d}"

check("compiled_matches", _test_compiled_matches, compiled_matches)

## Task 2: Find graph breaks

Python control flow on tensor **values** (like `if x.sum() > 0:`) or calls TorchDynamo cannot trace split the program into several graphs. Use `torch._dynamo.explain(fn)(*args)` and return `(graph_count, graph_break_count)` for `fn_branchy` defined below versus `fn_clean`.

In [ ]:
def fn_branchy(x):
    y = x * 2
    if y.sum() > 0:          # data-dependent branch
        y = y + 1
    print("side effect")     # also a break
    return y.relu()

def fn_clean(x):
    return (x * 2 + 1).relu()

def count_graph_breaks():
    # TODO: torch._dynamo.explain(fn)(torch.randn(8)) has .graph_count and .graph_break_count
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_count_graph_breaks(f):
    (gb, bb), (gc, bc) = f()
    assert bc == 0 and gc == 1, "the clean function is one graph"
    assert bb >= 1 and gb >= 2, "the branchy function breaks into several graphs"

check("count_graph_breaks", _test_count_graph_breaks, count_graph_breaks)

## Task 3: Memory of naive vs fused attention

Naive attention materialises the (B, H, T, T) score matrix; fused kernels (FlashAttention) never do. Return a dict for given B, H, T, D, bytes-per-element: `"scores"` = bytes of one score matrix, `"qkv"` = bytes of q, k and v together.
At long T, the scores dwarf q, k, v, which is the memory FlashAttention saves.

In [ ]:
def attention_bytes(B, H, T, D, bytes_per=2):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_attention_bytes(f):
    d = f(1, 32, 8192, 128)
    assert d["scores"] == 32 * 8192 * 8192 * 2 and d["qkv"] == 3 * 32 * 8192 * 128 * 2
    assert d["scores"] / d["qkv"] > 20

check("attention_bytes", _test_attention_bytes, attention_bytes)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Compile and compare with eager
def compiled_matches_solution():
    torch.manual_seed(0); x, w1, w2 = torch.randn(64, 128), torch.randn(128, 512) / 11, torch.randn(512, 128) / 22
    cf = torch.compile(gelu_mlp)
    return (cf(x, w1, w2) - gelu_mlp(x, w1, w2)).abs().max().item()

check("compiled_matches (reference)", _test_compiled_matches, compiled_matches_solution)

In [ ]:
# @title Solution for Task 2: Find graph breaks
def count_graph_breaks_solution():
    out = []
    for fn in (fn_branchy, fn_clean):
        torch._dynamo.reset()
        e = torch._dynamo.explain(fn)(torch.randn(8))
        out.append((e.graph_count, e.graph_break_count))
    return tuple(out)

check("count_graph_breaks (reference)", _test_count_graph_breaks, count_graph_breaks_solution)

In [ ]:
# @title Solution for Task 3: Memory of naive vs fused attention
def attention_bytes_solution(B, H, T, D, bytes_per=2):
    return {"scores": B * H * T * T * bytes_per, "qkv": 3 * B * H * T * D * bytes_per}

check("attention_bytes (reference)", _test_attention_bytes, attention_bytes_solution)

---
## Experiments
**Time it (CPU, rough).** Speedups on a laptop CPU are modest and noisy; on GPUs `torch.compile` fuses elementwise ops into Triton kernels and SDPA picks FlashAttention. The PyTorch 2 paper reports 2.27× inference and 1.41× training geometric-mean speedups on an A100 across 180+ models.

In [ ]:
torch.manual_seed(0); x, w1, w2 = torch.randn(256, 512), torch.randn(512, 2048) / 22, torch.randn(2048, 512) / 45
cf = torch.compile(gelu_mlp); cf(x, w1, w2)
def bench(fn, n=20):
    t = time.perf_counter()
    for _ in range(n): fn(x, w1, w2)
    return (time.perf_counter() - t) / n * 1e3
print(f"eager {bench(gelu_mlp):.2f} ms   compiled {bench(cf):.2f} ms   (CPU; numbers vary)")
for T in [1024, 8192, 131072]:
    d = attention_bytes_solution(1, 32, T, 128); print(f"T={T:6d}: score matrix {d['scores'] / 2**30:8.2f} GiB vs q,k,v {d['qkv'] / 2**30:6.3f} GiB")

---
## Takeaways
* `torch.compile` = TorchDynamo (captures Python bytecode into graphs) + AOTAutograd (adds the backward) + TorchInductor (generates fused Triton/C++ code). The default backend is "inductor".
* Graph breaks (data-dependent Python control flow, unsupported calls) split the graph and cost speed; find them with `torch._dynamo.explain`.
* `F.scaled_dot_product_attention` avoids materialising the T×T score matrix when a fused backend is available.